In [2]:
import sys
import os

# 1. Força o Notebook a apontar para a pasta raiz (FloodML) em vez da pasta experiments
if os.path.basename(os.getcwd()) == "experiments":
    os.chdir("../../")

# 2. Permite encontrar a pasta 'src'
sys.path.append(os.path.abspath(".")) 

import geemap
import ipywidgets as widgets
from IPython.display import display, clear_output

# Importando as funções da pasta src
from src.database.supabase_loader import inicializar_earth_engine, descarregar_dados_geograficos
from src.preprocessing.feature_engineering import gerar_features
from src.preprocessing.split import dividir_dados
from src.models.random_forest import treinar_modelo_rf
from src.models.evaluation import avaliar_modelo
from src.visualization.plots import calcular_rota_fuga, gerar_mapa_interativo
from src.visualization.metrics_plots import plotar_matriz_confusao

# Inicializar Earth Engine
inicializar_earth_engine()

Conexão com Earth Engine estabelecida.


In [3]:
Map = geemap.Map(center=[-15.7801, -47.9292], zoom=4)
output_area = widgets.Output()

def processar_desenho_automatico(target, action, geo_json):
    if action == 'created':
        with output_area:
            clear_output(wait=True) 
            print("\n" + "="*80)
            print("🚀 PIPELINE COMPLETO A EXECUTAR...")
            print("="*80)
            
            try:
                coords = geo_json['geometry']['coordinates'][0]
                
                # Passo 2: Extrair Dados
                relevo, agua, estruturas, regiao, limites = descarregar_dados_geograficos(coords)
                
                # Passo 4: Features
                X_estudo, y_gabarito, feature_relevo, fluxo = gerar_features(relevo, agua)
                
                # Split
                X_train, X_test, y_train, y_test = dividir_dados(X_estudo, y_gabarito)
                
                # Passos 3 e 5: Treino do Modelo
                pipeline = treinar_modelo_rf(X_train, y_train, X_test, y_test)
                
                # Passo 6: Avaliação
                y_pred_test = avaliar_modelo(pipeline, X_test, y_test)
                plotar_matriz_confusao(y_test, y_pred_test)
                
                # Passo 7: Rotas e Visualização Final
                rota_coords, mapa_previsao = calcular_rota_fuga(pipeline, X_estudo, relevo, estruturas, feature_relevo, limites)
                mapa_resultados = gerar_mapa_interativo(limites, relevo, agua, fluxo, mapa_previsao, estruturas, rota_coords)
                
                display(mapa_resultados)

            except Exception as e:
                print(f"❌ Erro crítico: {e}")

Map.draw_control.on_draw(processar_desenho_automatico)

print("🌍 AMBIENTE GEOGRÁFICO INTERATIVO")
print("1. Desenhe o retângulo na área desejada.")
display(Map)
display(output_area)

🌍 AMBIENTE GEOGRÁFICO INTERATIVO
1. Desenhe o retângulo na área desejada.


Map(center=[-15.7801, -47.9292], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

Output()